In [1]:
from sklearn.metrics import classification_report
from xgboost import XGBClassifier

from utils.bucketers import (
    PrefixLengthBucketer,
)
from utils.log_datasets import OutcomeDataset
from utils.pipelines import ProcessPredictorPipeline
from utils.stats import print_class_balance
from utils.transformers import (
    AggregateTransformer,
)

In [2]:
dataset_name = 'Traffic_Fines'
dataset_folder = '../datasets/raw'
labels_folder = '../datasets/labels/binary'

dataset = OutcomeDataset(
    dataset_name=dataset_name,
    dataset_folder=dataset_folder,
    labels_folder=labels_folder,
    feature_names=[
        'time_since_start',
        'time_since_last_event',
        'event_index',
        'day_of_week',
        'hour_of_day',
        'hour_sin',
        'hour_cos',
    ],
    min_prefix=3,
    max_prefix=10,
)

# Load raw data
dataset.load_and_preprocess()

# Filter out cases without labels
dataset.filter_by_labels()

dataset.raw_df.head()

Loading Traffic_Fines...


c:\Users\Pavel\Desktop\PROJECTS\master-thesis\.venv\lib\site-packages\pm4py\utils.py:990: UserWarning: In the current version, the import/export operation uses `rustxes` by default for importing/exporting files faster. Please uninstall `rustxes` to revert the behavior.
  warnings.warn(


,org:resource,lastSent,time:timestamp,case:concept:name,totalPaymentAmount,notificationType,points,paymentAmount,vehicleClass,lifecycle:transition,expense,amount,article,concept:name,matricola,dismissal
0,561,None,2006-07-23 22:00:00+00:00,A1,0.0,None,0.0,NaN,A,complete,NaN,35.0,157.0,Create Fine,NaN,NIL
1,None,None,2006-12-04 23:00:00+00:00,A1,NaN,None,NaN,NaN,None,complete,11.0,NaN,NaN,Send Fine,NaN,None
2,561,None,2006-08-01 22:00:00+00:00,A100,0.0,None,0.0,NaN,A,complete,NaN,35.0,157.0,Create Fine,NaN,NIL
3,None,None,2006-12-11 23:00:00+00:00,A100,NaN,None,NaN,NaN,None,complete,11.0,NaN,NaN,Send Fine,NaN,None
4,None,P,2007-01-14 23:00:00+00:00,A100,NaN,P,NaN,NaN,None,complete,NaN,NaN,NaN,Insert Fine Notification,NaN,None


In [3]:
# Split into train/test
train_df, test_df = dataset.train_test_split()

train_df.head()

Train cases: 120296, Test cases: 30074


,event_index,day_of_week,case:concept:name,concept:name,hour_sin,time_since_last_event,hour_cos,hour_of_day,time:timestamp,time_since_start
0,1,6,A1,Create Fine,-0.500000,0.000000,0.866025,22,2006-07-23 22:00:00+00:00,0.000000
1,2,0,A1,Send Fine,-0.258819,134.041667,0.965926,23,2006-12-04 23:00:00+00:00,134.041667
2,1,1,A100,Create Fine,-0.500000,0.000000,0.866025,22,2006-08-01 22:00:00+00:00,0.000000
3,2,0,A100,Send Fine,-0.258819,132.041667,0.965926,23,2006-12-11 23:00:00+00:00,132.041667
4,3,6,A100,Insert Fine Notification,-0.258819,34.000000,0.965926,23,2007-01-14 23:00:00+00:00,166.041667


In [4]:
# Generate prefixes
train_prefixes = dataset.get_prefixes(train_df)
test_prefixes = dataset.get_prefixes(test_df)

# Prepare labels
y_train = dataset.prepare_labels(train_prefixes)
y_test = dataset.prepare_labels(test_prefixes)

# Or directly from the original dataframes
# y_train = dataset.prepare_labels(train_df)
# y_test = dataset.prepare_labels(test_df)

train_prefixes.head()

Label encoding: {0: 'Other', 1: 'Repaid'}


,event_index,day_of_week,case:concept:name,concept:name,hour_sin,time_since_last_event,hour_cos,hour_of_day,time:timestamp,time_since_start,prefix_id,prefix_len
0,1,1,A100,Create Fine,-0.500000,0.000000,0.866025,22,2006-08-01 22:00:00+00:00,0.000000,A100_prefix_3,3
1,2,0,A100,Send Fine,-0.258819,132.041667,0.965926,23,2006-12-11 23:00:00+00:00,132.041667,A100_prefix_3,3
2,3,6,A100,Insert Fine Notification,-0.258819,34.000000,0.965926,23,2007-01-14 23:00:00+00:00,166.041667,A100_prefix_3,3
3,1,1,A100,Create Fine,-0.500000,0.000000,0.866025,22,2006-08-01 22:00:00+00:00,0.000000,A100_prefix_4,4
4,2,0,A100,Send Fine,-0.258819,132.041667,0.965926,23,2006-12-11 23:00:00+00:00,132.041667,A100_prefix_4,4


In [5]:
print_class_balance(y_train, dataset_names=[dataset_name, 'Train'])
print_class_balance(y_test, dataset_names=[dataset_name, 'Test'])


            Class Balance - Traffic_Fines, Train            
Class                                 Count   Percentage
------------------------------------------------------------
0                                   607,385       68.39%
1                                   280,735       31.61%
------------------------------------------------------------
Total                               888,120      100.00%


            Class Balance - Traffic_Fines, Test             
Class                                 Count   Percentage
------------------------------------------------------------
0                                   124,346       65.11%
1                                    66,622       34.89%
------------------------------------------------------------
Total                               190,968      100.00%



In [6]:
# Pipeline params
bucketer = PrefixLengthBucketer(case_id_col='prefix_id')
transformer = AggregateTransformer(
    case_id_col='prefix_id',
    cat_cols=['concept:name'],
    num_cols=['time_since_last_event', 'time_since_start'],
)

num_classes = len(dataset.label_encoder.classes_)

if num_classes == 2:
    # Binary classification
    model = XGBClassifier(n_estimators=100, max_depth=5, objective='binary:logistic')
else:
    # Multiclass classification
    model = XGBClassifier(
        n_estimators=100, max_depth=5, objective='multi:softprob', num_class=num_classes
    )

In [7]:
# Build and fit the Pipeline
pipeline = ProcessPredictorPipeline(bucketer, transformer, model)
pipeline.fit(train_prefixes, y_train)

Training bucket: 3...
Training bucket: 4...
Training bucket: 5...
Training bucket: 6...
Training bucket: 7...
Training bucket: 8...
Training bucket: 9...
Training bucket: 10...


In [8]:
# Evaluate
y_pred = pipeline.predict(test_prefixes)

# Decode predictions and labels for evaluation
y_pred_decoded = dataset.decode_labels(y_pred)
y_test_decoded = dataset.decode_labels(y_test)

print(classification_report(y_test_decoded, y_pred_decoded))

              precision    recall  f1-score   support

       Other       0.81      0.97      0.88    124346
      Repaid       0.90      0.58      0.70     66622

    accuracy                           0.83    190968
   macro avg       0.86      0.77      0.79    190968
weighted avg       0.84      0.83      0.82    190968



In [14]:
for bucket, model in pipeline.bucket_models.items():
    print(f'Bucket {bucket} model:')
    print(model)

Bucket 3 model:
XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=100,
              n_jobs=None, num_parallel_tree=None, ...)
Bucket 4 model:
XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
           

[Outcome-Oriented Predictive Process Monitoring: Review and Benchmark](https://dl.acm.org/doi/pdf/10.1145/3301300)

Datasets:

- BPIC15_1
- BPIC15_2
- BPIC15_3
- BPIC15_4
- BPIC15_5
- Sepsis
- Traffic_Fines

Data Encodings:

- Static encoding
- Last state encoding
- Aggregation encoding
- Index-based encoding

Bucketing strategies:

- Single Bucket
- KNN
- Clustering
- Prefix Length

Methods:

- XGBoost
- RF
- Logistic Regression
- SVM

Input features can be: activity, timestamp, resource and attributes

[Features Transformers](https://github.com/nirdizati/nirdizati-training-backend/tree/master/transformers)
